# Final Project

Author: Evan Whitfield

Course: ST 554

Purpose: Final Project

## Part 1 - Fitting Your Model

First, we need to import all the necessary modules for our task.

In [1]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator


We need to read in our data for our machine learning process. This will serve as our training set for our model building process.

In [2]:
power_ml_data = pd.read_csv('power_ml_data.csv')

Time to start the spark session and turn the data into a spark dataframe.

In [3]:
spark_sess = SparkSession.builder.getOrCreate()

spark_df = spark_sess.createDataFrame(power_ml_data)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 17:12:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 17:12:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 17:12:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
spark_df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [5]:
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



We want the `Hour` variable to be a Double type instead of a Long type. 

In [6]:
cast_hour_double = SQLTransformer(statement = 
    """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

Next, we want to change our hour variable into a binary for either day or night time using a threshold of 6.5.

In [7]:
binary_day_night = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="day_night")

We want to One Hot Encode the month variable.

In [16]:
one_hot = OneHotEncoder(
    inputCols = ["Month"],
    outputCols = ["Month_ohe"],
    dropLast = False
)

Time to assemble the different weather related variables into a single column.

In [17]:
weather_assembled = VectorAssembler(
    inputCols = ["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"],
    outputCol = "weather",
    handleInvalid = "skip"
)

The next transform we want to do is a PCA with k = 2 and store the results in a new column.

In [18]:
pca = PCA(
    k = 2,
    inputCol = "weather",
    outputCol = "pca_features"
)

We need to set up our label and features column to get ready for the Linear Regression.

In [11]:
label_maker = SQLTransformer(statement =
    """
    SELECT *, Power_Zone_3 AS label
    FROM __THIS__
    """
)

In [19]:
features_assembler = VectorAssembler(
    inputCols = ["pca_features","day_night","Power_Zone_1","Power_Zone_2","Month_ohe"],
    outputCol = "features",
    handleInvalid = "skip"
)

In [13]:
lr = LinearRegression()

Next, we want to set up our grid of possible parameters to test to check to see what will produce the lowest RMSE.

In [14]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

Time to put all of our different transforms into a pipeline, ending with the Linear Regression.

In [15]:
pipeline = Pipeline(stages = [
    cast_hour_double, 
    binary_day_night, 
    one_hot, 
    weather_assembled,
    pca,
    label_maker,
    features_assembler,
    lr
])

The pipeline will be used as our estimator with 5 folds. We want to determine which set of parameters gives us the lowest RMSE.

In [20]:
cv = CrossValidator(
    estimator = pipeline,
    estimatorParamMaps = paramGrid,
    evaluator = RegressionEvaluator(metricName = 'rmse'),
    numFolds = 5,
    parallelism = 8
)

In [21]:
cvModel = cv.fit(spark_df)

26/04/30 17:15:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 17:15:34 WARN Instrumentation: [fa1578c7] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [546b2287] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [c7e87908] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [3ae85171] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [1d631d1e] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [42d1c95f] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 17:15:34 WARN Instrumentation: [0d951d23] regP

We need to do use a couple of attributes to reside to produce the best parameters for reducing the RMSE.

In [25]:
best_pipeline_model = cvModel.bestModel.stages[-1]

print("Best regParam:", best_pipeline_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_pipeline_model._java_obj.getElasticNetParam())

Best regParam: 0.25
Best elasticNetParam: 0.95


In [26]:
rmse_values = cvModel.avgMetrics
best_rmse = min(rmse_values)

print("Best CV RMSE:", best_rmse)

Best CV RMSE: 2148.0540674676627


In [27]:
preds = cvModel.transform(spark_df).select("label", "features", "prediction")

In [33]:
training_error = RegressionEvaluator().evaluate(cvModel.transform(spark_df))
print("CV with training error:", training_error)

CV with training error: 2147.098183431141


In [35]:
preds_with_resids = preds.withColumn("residual",
    col("label") - col("prediction")
)

In [36]:
preds_with_resids.show(20)

+-----------+--------------------+------------------+------------------+
|      label|            features|        prediction|          residual|
+-----------+--------------------+------------------+------------------+
|20240.96386|(18,[0,1,3,4,6],[...| 20877.03715122213| -636.073291222132|
|20131.08434|(18,[0,1,3,4,6],[...| 18657.38668217668|1473.6976578233225|
|19668.43373|(18,[0,1,3,4,6],[...|18202.010613483697|1466.4231165163037|
|18899.27711|(18,[0,1,3,4,6],[...|17588.085062241633|1311.1920477583662|
|18442.40964|(18,[0,1,3,4,6],[...| 16994.87027644466| 1447.539363555341|
|18130.12048|(18,[0,1,3,4,6],[...|16515.379689021524| 1614.740790978478|
|17945.06024|(18,[0,1,3,4,6],[...|16091.055327862201|1854.0049121377979|
|17459.27711|(18,[0,1,3,4,6],[...|15720.594771047288|1738.6823389527108|
|17025.54217|(18,[0,1,3,4,6],[...|15269.062017847804|1756.4801521521968|
|16794.21687|(18,[0,1,3,4,6],[...|14936.444981823015|1857.7718881769852|
|16638.07229|(18,[0,1,3,4,6],[...|14650.62360171661

## Part 2 - Streaming Part

In [37]:
schema = spark_df.schema

In [38]:
stream_df = spark_sess.readStream \
    .format("csv") \
    .option("header", True) \
    .schema(schema) \
    .load("Final Project Output")

In [39]:
stream_preds = cvModel.transform(stream_df)

stream_residuals = stream_preds.withColumn(
    "residual", col("label") - col("prediction")
    ).select(
        "label",
        "prediction",
        "residual"
)

In [40]:
label_stream = SQLTransformer(statement = 
    """
    SELECT Power_Zone_3 AS label
    FROM __THIS__
    """
    ).transform(stream_df)

In [41]:
joined_stream = stream_residuals.join(
    label_stream,
    on = "label",
    how = "inner"
)

In [42]:
query = joined_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/04/30 17:25:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-58938e0f-6db7-4fa7-b8b4-8155125d5362. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 17:25:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
| 17493.9759|14307.384965494757|  3186.590934505244|
|15560.16194|15965.563938151188| -405.4019981511883|
|15423.61446|16789.239727258046| -1365.625267258045|
|24154.64435|24006.745113218054| 147.89923678194464|
|12997.81818|13233.388712083759|-235.57053208375874|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|10673.79939|10089.940879199175|  583.8585108008247|
|10403.85542|12389.887910664593|-1986.0324906645928|
|23931.98381| 22978.03501196836|  953.9487980316408|
|12161.94529|15407.167684437345|-3245.2223944373454|
| 34216.9279|31808.780610465084|  2408.147289534918|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|9494.954407|10051.698723505242| -556.7443165052428|
|17719.51807|11906.529309242896|  5812.988760757102|
|11126.74699| 9799.682060527726| 1327.0649294722734|
|25320.84422|25696.045136578337|-375.20091657833837|
|15921.61943|15951.386600191641|-29.767170191640616|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
| 18014.5749|18254.732964125138|-240.15806412513848|
|25017.83133|24134.235794504526|  883.5955354954749|
|15020.71502|15238.815501211686| -218.1004812116862|
|11907.46988| 11928.45852484425|-20.988644844250302|
|10854.93976| 10513.05757221378|  341.8821877862192|
+-----------+------------------+-------------------+



...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|15067.78345|11436.848763459018|3630.9346865409825|
|14350.39514|12238.675579600358|2111.7195603996424|
|16487.71084|17813.954820493433|-1326.243980493433|
|13659.59514|12419.620888735242|1239.9742512647572|
|9368.674699|10236.400364083096|-867.7256650830968|
+-----------+------------------+------------------+



-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|17517.10843| 17307.46068081059|209.64774918941112|
|19656.86747| 16728.99965955734|2927.8678104426617|
|15245.34413|15191.045351485182| 54.29877851481797|
|11077.81818|11817.120409461253|-739.3022294612529|
|22044.30151| 20870.12906538795|1174.1724446120497|
+-----------+------------------+------------------+



-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|12465.23077|14937.761793985415|-2472.5310239854152|
|10687.01538| 14222.24713056427|-3535.2317505642695|
|18170.18182|18805.110852900463| -634.9290329004616|
|17954.90909|16734.580108145397| 1220.3289818546036|
|14161.71604|11852.891128929039| 2308.8249110709603|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|25257.16583|24657.150867676508|  600.0149623234938|
|15857.48988|16228.420003211986|-370.93012321198694|
|13002.83077|15721.575833216617|-2718.7450632166165|
|20631.27273|19049.794522686338|  1581.478207313663|
|22245.51724|22585.887082986348|-340.36984298634707|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|22923.87097|23220.574410621175| -296.7034406211751|
|26526.31579|26892.938534856054|-366.62274485605303|
| 15093.9759|14960.370132824015|  133.6057671759845|
|18540.54711| 20609.29516723358|-2068.7480572335808|
|10108.91566|10674.700401297507| -565.7847412975061|
+-----------+------------------+-------------------+



In [79]:
query.stop()

26/04/29 23:14:20 WARN DAGScheduler: Failed to cancel job group e4543d50-030b-4dec-a7c5-a28420b6119f. Cannot find active jobs for it.
26/04/29 23:14:21 WARN DAGScheduler: Failed to cancel job group e4543d50-030b-4dec-a7c5-a28420b6119f. Cannot find active jobs for it.
